![Banner Fast Track](https://upload.wikimedia.org/wikipedia/commons/thumb/b/b1/Camara_dos_Deputados_-_Plen%C3%A1rio.jpg/1200px-Camara_dos_Deputados_-_Plen%C3%A1rio.jpg)

# 🏛️ Fast Track — 00. Inicialização do Pipeline

**Pipeline de Dados Completo — Câmara dos Deputados (Padrão Petrobras)**

| Item | Descrição |
|---|---|
| **Objetivo** | Inicializar toda a infraestrutura do pipeline Fast Track: catálogo Unity Catalog, schemas (bronze/silver/gold/ops/qa/log), tabelas de controle, geração de load_id e registro da execução |
| **Arquitetura** | Medalhão (Bronze → Silver → Gold) + Ops/QA/Log sobre Delta Lake no Unity Catalog |
| **Catálogo UC** | `uc_fast_track` |
| **Schemas** | `bronze` (raw), `silver` (conform), `gold` (marts), `ops` (controle operacional), `qa` (qualidade), `log` (troubleshooting detalhado) |
| **Padrão** | Petrobras (prático) — workflows parametrizados, checkpoints, replay garantido |
| **Fonte** | https://dadosabertos.camara.leg.br/api/v2 |
| **Responsável** | Ernesto Bassoli |
| **Programa** | Upskill Tiller – Engenharia de Dados (T2.7) |

---

## 📋 O Que Este Notebook Faz

Este é o **PRIMEIRO notebook executado** em qualquer execução do pipeline Fast Track. Ele é responsável por:

1. **Criar toda a infraestrutura necessária** (catálogo Unity Catalog, 6 schemas, tabelas de controle)
2. **Gerar um Load ID único** (UUID) que identifica esta execução específica
3. **Registrar o início da execução** em tabelas operacionais para rastreabilidade
4. **Carregar funções utilitárias** que serão usadas por todos os outros notebooks
5. **Validar que tudo foi criado corretamente** antes de iniciar a ingestão

**IMPORTANTE:** Este notebook é **idempotente** — pode ser executado múltiplas vezes sem problema. Todas as operações usam `CREATE IF NOT EXISTS`.

---

## 🔄 Quando Este Notebook É Executado

* **Workflow A (Diário)**: Primeira task (A00_init_run)
* **Workflow B (Micro-batch 10 min)**: Primeira task (B00_init_run_microbatch)
* **Manualmente**: Para criar/recriar infraestrutura ou testar

---

## 🎯 Entregas Deste Notebook

Ao final da execução, você terá:

✅ **Catálogo UC**: `uc_fast_track` criado
✅ **6 Schemas**: bronze, silver, gold, ops, qa, log
✅ **13 Tabelas de Controle**: ops (4), qa (3), log (6)
✅ **Load ID**: UUID único gerado e exibido
✅ **Registro Inicial**: 1 linha em `ops.ingestion_runs` com status RUNNING
✅ **Funções Carregadas**: 5 funções utilitárias disponíveis para outros notebooks

## ⚙️ Bloco 1: Configuração do Ambiente

### 📋 Descrição

Este bloco configura os **parâmetros de execução** do pipeline através de widgets interativos. Estes parâmetros são fundamentais para controlar o comportamento de TODO o pipeline.

---

### 🎛️ Parâmetros (Widgets)

#### 1. **`env`** (Ambiente)
**Valores possíveis:** `dev`, `hml`, `prd`

**O que faz:**
* Define o ambiente de execução (desenvolvimento, homologação ou produção)
* Afeta configurações como: limites de API, timeouts, volumes processados
* **Dev**: permite execuções parciais e testes
* **Hml**: espelha produção mas em ambiente isolado
* **Prd**: execução completa e oficial

**Onde é usado:** Registrado em todas as tabelas de log/controle para rastreabilidade

---

#### 2. **`run_date`** (Data de Execução)
**Formato:** `YYYY-MM-DD` (ex: `2026-04-27`)

**O que faz:**
* Define a data de referência desta execução
* **Não é necessariamente hoje** — permite reprocessamento de datas passadas
* Usado para: filtrar dados da API, particionar tabelas, calcular janelas temporais

**Exemplo:**
* Executando hoje (27/04) com `run_date=2026-04-20` → reprocessa dados de 20/04

---

#### 3. **`full_refresh`** (Recarga Completa)
**Valores possíveis:** `true`, `false`

**O que faz:**
* **`false` (padrão)**: Modo incremental — processa apenas dados novos desde o último checkpoint
* **`true`**: Modo full refresh — reprocessa TUDO do zero, ignorando checkpoints

**Quando usar `true`:**
* Primeira execução (não há checkpoints ainda)
* Correção de erros históricos (reprocessar tudo)
* Mudança de lógica de negócio (recalcular tudo)

**Cuidado:** `full_refresh=true` pode demorar muito e sobrecarregar a API!

---

### 🔧 Variáveis Globais

Além dos widgets, este bloco define variáveis globais usadas em todo o pipeline:

* **`catalog`**: Nome do catálogo Unity Catalog (`uc_fast_track`)
* **`schemas`**: Lista com os 6 schemas (bronze, silver, gold, ops, qa, log)

---

### 💡 Para Leigos

Imagine que você está dirigindo um carro:
* **`env`** = Modo de direção (econômico/normal/esportivo)
* **`run_date`** = Data da viagem (pode ser hoje ou reagendar uma viagem antiga)
* **`full_refresh`** = Limpar tudo e começar do zero vs. continuar de onde parou

Estes parâmetros dão controle total sobre como o pipeline vai rodar!

In [0]:
# ============================================================
# CONFIGURAÇÃO DO AMBIENTE E PARÂMETROS DE EXECUÇÃO
# ============================================================
# Define os widgets (parâmetros interativos) que controlam
# como o pipeline será executado e as variáveis globais.
# ============================================================

# ============================================================
# PASSO 1: Criar Widgets de Parametrização
# ============================================================
# Widgets permitem passar parâmetros quando o notebook é
# executado via workflow ou manualmente.

# Widget 1: Ambiente (dev, hml, prd)
# Define se estamos em desenvolvimento, homologação ou produção.
dbutils.widgets.text("env", "dev", "Ambiente (dev/hml/prd)")

# Widget 2: Data de Execução (YYYY-MM-DD)
# Define a data de referência desta execução.
# Padrão: data de hoje.
from datetime import datetime
data_hoje = datetime.now().strftime("%Y-%m-%d")
dbutils.widgets.text("run_date", data_hoje, "Data de Execução (YYYY-MM-DD)")

# Widget 3: Full Refresh (true/false)
# Define se deve reprocessar tudo do zero ou apenas incremental.
dbutils.widgets.dropdown("full_refresh", "false", ["false", "true"], "Full Refresh?")

# ============================================================
# PASSO 2: Recuperar Valores dos Widgets
# ============================================================
# Lê os valores que foram passados (ou usa os padrões).

env = dbutils.widgets.get("env")
run_date = dbutils.widgets.get("run_date")
full_refresh = dbutils.widgets.get("full_refresh") == "true"  # Converte string para boolean

# ============================================================
# PASSO 3: Definir Variáveis Globais do Pipeline
# ============================================================
# Estas variáveis serão usadas em TODOS os notebooks do pipeline.

# Nome do catálogo Unity Catalog
catalog = "uc_fast_track"

# Nomes dos schemas (um para cada camada)
schema_bronze = "bronze"
schema_silver = "silver"
schema_gold = "gold"
schema_ops = "ops"      # Operacional (checkpoints, logs)
schema_qa = "qa"        # Qualidade (regras QA, rejects)
schema_log = "log"      # Logs detalhados por camada

# ============================================================
# PASSO 4: Exibir Configurações para Confirmação
# ============================================================
# Mostra no console os valores configurados para validação.

print("="*70)
print("⚙️  CONFIGURAÇÃO DO PIPELINE FAST TRACK")
print("="*70)
print()
print(f"📍 Ambiente (env):              {env}")
print(f"📅 Data de Execução (run_date): {run_date}")
print(f"🔄 Full Refresh:                {full_refresh}")
print()
print(f"🗄️  Catálogo UC:                 {catalog}")
print(f"📂 Schemas:")
print(f"   • Bronze (raw):              {catalog}.{schema_bronze}")
print(f"   • Silver (conform):          {catalog}.{schema_silver}")
print(f"   • Gold (marts):              {catalog}.{schema_gold}")
print(f"   • Ops (controle):            {catalog}.{schema_ops}")
print(f"   • QA (qualidade):            {catalog}.{schema_qa}")
print(f"   • Log (detalhado):           {catalog}.{schema_log}")
print()
print("✅ Configuração carregada com sucesso!")
print("="*70)

## 📦 Bloco 2: Importações e Dependências

### 📋 Descrição

Este bloco importa todas as bibliotecas Python e PySpark necessárias para o pipeline funcionar. Cada biblioteca tem um propósito específico.

---

### 📚 Bibliotecas Importadas

#### **Bibliotecas Python Nativas**

* **`requests`**: Faz chamadas HTTP para a API da Câmara dos Deputados
* **`json`**: Manipula dados JSON (leitura, escrita, conversão)
* **`time`**: Controla pausas entre requisições (evita rate limiting da API)
* **`datetime`**: Manipula datas e timestamps (campos de auditoria)
* **`uuid`**: Gera IDs únicos (load_id para cada execução)
* **`hashlib`**: Calcula hashes SHA256 dos payloads (deduplicação e CDC)

---

#### **PySpark — Tipos de Dados**

Usado para definir schemas explícitos das tabelas Delta:

* **`StructType`**: Define a estrutura completa de uma tabela
* **`StructField`**: Define uma coluna individual
* **Tipos**: `StringType`, `IntegerType`, `LongType`, `DoubleType`, `BooleanType`, `TimestampType`, `DateType`

**Por que definir schemas explícitos?**
* Performance: Databricks otimiza melhor quando sabe os tipos
* Validação: Erros de tipo são detectados cedo
* Documentação: Schema é auto-documentado

---

#### **PySpark — Funções de Transformação**

Usado para manipular colunas em DataFrames:

* **`col`**: Referencia uma coluna pelo nome
* **`lit`**: Cria uma coluna com valor literal (constante)
* **`current_timestamp`**: Retorna o timestamp atual
* **`to_date`**, **`to_timestamp`**: Converte strings para datas/timestamps
* **`sha2`**: Calcula hash diretamente em uma coluna Spark
* **`when`**, **`coalesce`**: Lógica condicional

---

### 💡 Para Leigos

Pense nas bibliotecas como **ferramentas em uma caixa de ferramentas**:

* **`requests`** = Telefone (para chamar a API)
* **`json`** = Tradutor (converte JSON para Python e vice-versa)
* **`uuid`** = Gerador de etiquetas únicas (identifica cada execução)
* **`hashlib`** = Impressão digital (identifica se um dado mudou)
* **PySpark** = Guincho (processa milhões de linhas em paralelo)

Sem essas ferramentas, o pipeline não funciona!

In [0]:
# ============================================================
# IMPORTAÇÕES DE BIBLIOTECAS
# ============================================================
# Importa todas as dependências necessárias para o pipeline.
# ============================================================

print("📦 Carregando bibliotecas...")

# ============================================================
# BIBLIOTECAS PYTHON NATIVAS
# ============================================================

# Para fazer chamadas HTTP à API da Câmara dos Deputados
import requests

# Para manipular dados JSON (parse, dumps, loads)
import json

# Para controlar tempo entre requisições (evitar rate limiting)
import time

# Para manipular datas e timestamps
from datetime import datetime, timedelta, date

# Para gerar IDs únicos de execução (load_id)
import uuid

# Para calcular hashes SHA256 dos payloads (deduplicação)
import hashlib

# ============================================================
# PYSPARK — TIPOS DE DADOS
# ============================================================
# Usados para definir schemas explícitos das tabelas Delta.

from pyspark.sql.types import (
    StructType,       # Define estrutura completa de uma tabela
    StructField,      # Define uma coluna individual
    StringType,       # Texto (ex: nomes, descrições)
    IntegerType,      # Número inteiro pequeno (ex: IDs, contadores)
    LongType,         # Número inteiro grande (ex: timestamps em ms)
    DoubleType,       # Número decimal (ex: valores monetários)
    BooleanType,      # Verdadeiro/Falso
    TimestampType,    # Data + Hora (ex: 2026-04-27 21:30:45)
    DateType,         # Apenas Data (ex: 2026-04-27)
    ArrayType,        # Lista de valores
    MapType           # Dicionário (chave-valor)
)

# ============================================================
# PYSPARK — FUNÇÕES DE TRANSFORMAÇÃO
# ============================================================
# Usadas para manipular colunas em DataFrames.

from pyspark.sql.functions import (
    col,                    # Referencia uma coluna pelo nome
    lit,                    # Cria coluna com valor literal/constante
    current_timestamp,      # Retorna timestamp atual
    current_date,           # Retorna data atual
    to_date,                # Converte string para data
    to_timestamp,           # Converte string para timestamp
    date_format,            # Formata data como string
    year, month, day,       # Extrai componentes de data
    sha2,                   # Calcula hash SHA2/SHA256
    md5,                    # Calcula hash MD5
    when,                   # Lógica condicional (IF-THEN-ELSE)
    coalesce,               # Retorna primeiro valor não-nulo
    concat,                 # Concatena strings
    trim,                   # Remove espaços em branco
    upper, lower,           # Converte para maiúscula/minúscula
    regexp_replace,         # Substitui padrões com regex
    split,                  # Divide string em array
    explode,                # "Explode" array em múltiplas linhas
    size,                   # Tamanho de array/map
    count, sum, avg, max, min  # Agregações
)

# ============================================================
# VALIDAÇÃO DAS IMPORTAÇÕES
# ============================================================
# Testa se as bibliotecas foram importadas corretamente.

try:
    # Testa se o SparkSession está disponível
    spark
    print("✅ SparkSession disponível")
except NameError:
    print("❌ ERRO: SparkSession não encontrado!")
    raise

print("✅ Todas as bibliotecas carregadas com sucesso!")
print(f"   • requests {requests.__version__}")
print(f"   • PySpark (via SparkSession)")
print(f"   • Python {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 🗄️ Bloco 3: Criação do Catálogo e Schemas

### 📋 Descrição

Este bloco cria a **estrutura fundamental** do Unity Catalog onde todos os dados do pipeline serão armazenados.

---

### 🏗️ Hierarquia do Unity Catalog

```
uc_fast_track (CATÁLOGO)
├── bronze (SCHEMA)    → Dados brutos da API
├── silver (SCHEMA)    → Dados limpos e tipados
├── gold (SCHEMA)      → Marts analíticos
├── ops (SCHEMA)       → Controle operacional
├── qa (SCHEMA)        → Qualidade de dados
└── log (SCHEMA)       → Logs detalhados
```

---

### 📂 Schemas e Suas Funções

#### **1. `bronze` (Camada Raw)**
**Função:** Armazenar dados **exatamente como vieram** da API.

**Características:**
* Payload JSON completo como STRING
* Colunas técnicas (ingestion_ts, load_id, record_hash)
* Mode APPEND (nunca deleta histórico)
* Particionado por `ingestion_date`

**Exemplo de tabela:** `bronze.deputados_raw`, `bronze.ceap_despesas_raw`

---

#### **2. `silver` (Camada Conform)**
**Função:** Dados **limpos, tipados e relacionados**.

**Características:**
* Parse do JSON em colunas tipadas (INT, STRING, DATE, etc)
* Deduplicação
* Padronização (nomes, datas, valores)
* Chaves estrangeiras explícitas
* SCD Type 2 quando necessário

**Exemplo de tabela:** `silver.deputados`, `silver.frentes_membros`

---

#### **3. `gold` (Camada Marts)**
**Função:** Tabelas/views **prontas para análise e dashboards**.

**Características:**
* Agregações pré-calculadas
* Joins entre múltiplas entidades
* Métricas de negócio
* Otimizadas para consulta (Z-ORDER, particionamento)

**Exemplo de tabela:** `gold.mart_frentes`, `gold.mart_ceap_anomalias`

---

#### **4. `ops` (Controle Operacional)**
**Função:** Tabelas que **controlam o funcionamento** do pipeline.

**Tabelas principais:**
* `ops.ingestion_runs`: Histórico de execuções (1 linha por run)
* `ops.checkpoints`: Controle incremental (último ID/data processado)
* `ops.ingestion_requests`: Log de chamadas HTTP à API
* `ops.dead_letter`: Payloads que falharam

---

#### **5. `qa` (Qualidade de Dados)**
**Função:** Armazenar **resultados de regras de qualidade**.

**Tabelas principais:**
* `qa.quality_results`: Resultado de cada regra QA (PASS/FAIL)
* `qa.quality_metrics`: Métricas agregadas (volume, completude, etc)
* `qa.rejects`: Registros rejeitados com motivo

---

#### **6. `log` (Logs Detalhados)**
**Função:** Logs **granulares por camada** para troubleshooting.

**Tabelas principais:**
* `log.bronze_ingest_runs`: Log de cada ingestão Bronze
* `log.silver_transform_runs`: Log de cada transformação Silver
* `log.gold_mart_runs`: Log de cada criação de mart Gold
* `log.bronze_api_requests`: Log HTTP detalhado
* `log.quality_checks`: Histórico completo de QA

---

### 💡 Para Leigos

Pense no Unity Catalog como um **prédio de escritórios**:

* **Catálogo** = Prédio inteiro (`uc_fast_track`)
* **Schema** = Andar do prédio (bronze = térreo, silver = 1º andar, gold = cobertura)
* **Tabela** = Sala dentro do andar

Cada andar tem uma função específica, e os dados "sobem" do térreo (bronze) até a cobertura (gold) ficando cada vez mais refinados!

In [0]:
# ============================================================
# CRIAÇÃO DO CATÁLOGO UNITY CATALOG
# ============================================================
# Cria o catálogo principal onde todos os dados serão armazenados.
# IMPORTANTE: Operação idempotente (pode executar múltiplas vezes).
# ============================================================

print("🗄️  Criando catálogo Unity Catalog...")
print()

# ============================================================
# PASSO 1: Criar Catálogo (se não existir)
# ============================================================
# IF NOT EXISTS garante que não dá erro se já existir.

sql_create_catalog = f"""
CREATE CATALOG IF NOT EXISTS {catalog}
COMMENT 'Catálogo Fast Track - Pipeline Câmara dos Deputados (Padrão Petrobras)'
"""

# Executa o comando SQL
spark.sql(sql_create_catalog)

print(f"✅ Catálogo '{catalog}' criado/validado")
print(f"   Contém todos os dados do pipeline Fast Track")
print()

# ============================================================
# PASSO 2: Definir Catálogo Padrão
# ============================================================
# Configura o catálogo padrão para evitar ter que especificar
# em todas as queries.

spark.sql(f"USE CATALOG {catalog}")

print(f"✅ Catálogo '{catalog}' definido como padrão")
print(f"   Todas as queries usarão este catálogo automaticamente")
print()
print("="*70)

In [0]:
# ============================================================
# CRIAÇÃO DOS 6 SCHEMAS DO PIPELINE
# ============================================================
# Cria os schemas (camadas) onde as tabelas serão organizadas.
# ============================================================

print("📂 Criando schemas (camadas do pipeline)...")
print()

# ============================================================
# SCHEMA 1: BRONZE (Dados Raw)
# ============================================================
# Armazena dados brutos exatamente como vieram da API.

sql_bronze = f"""
CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_bronze}
COMMENT 'Camada Bronze - Dados brutos da API (payload JSON raw + colunas técnicas)'
"""
spark.sql(sql_bronze)
print(f"✅ Schema '{schema_bronze}' criado")
print(f"   → Dados raw da API com payload JSON completo")
print()

# ============================================================
# SCHEMA 2: SILVER (Dados Conform)
# ============================================================
# Armazena dados limpos, tipados e relacionados.

sql_silver = f"""
CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_silver}
COMMENT 'Camada Silver - Dados limpos, tipados e relacionados (parse do JSON, deduplicação, SCD2)'
"""
spark.sql(sql_silver)
print(f"✅ Schema '{schema_silver}' criado")
print(f"   → Dados limpos e tipados com relacionamentos explícitos")
print()

# ============================================================
# SCHEMA 3: GOLD (Marts Analíticos)
# ============================================================
# Armazena agregações e marts prontos para análise.

sql_gold = f"""
CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_gold}
COMMENT 'Camada Gold - Marts analíticos (agregações, joins, métricas de negócio)'
"""
spark.sql(sql_gold)
print(f"✅ Schema '{schema_gold}' criado")
print(f"   → Marts analíticos prontos para dashboards")
print()

# ============================================================
# SCHEMA 4: OPS (Controle Operacional)
# ============================================================
# Armazena checkpoints, logs de execução e dead letter.

sql_ops = f"""
CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_ops}
COMMENT 'Schema Ops - Controle operacional (ingestion_runs, checkpoints, dead_letter)'
"""
spark.sql(sql_ops)
print(f"✅ Schema '{schema_ops}' criado")
print(f"   → Controle operacional (checkpoints, runs, requests)")
print()

# ============================================================
# SCHEMA 5: QA (Qualidade de Dados)
# ============================================================
# Armazena resultados de regras QA e registros rejeitados.

sql_qa = f"""
CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_qa}
COMMENT 'Schema QA - Qualidade de dados (quality_results, quality_metrics, rejects)'
"""
spark.sql(sql_qa)
print(f"✅ Schema '{schema_qa}' criado")
print(f"   → Qualidade de dados (regras QA, métricas, rejects)")
print()

# ============================================================
# SCHEMA 6: LOG (Logs Detalhados)
# ============================================================
# Armazena logs granulares por camada para troubleshooting.

sql_log = f"""
CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_log}
COMMENT 'Schema Log - Logs detalhados por camada (bronze_ingest, silver_transform, gold_marts)'
"""
spark.sql(sql_log)
print(f"✅ Schema '{schema_log}' criado")
print(f"   → Logs detalhados por camada para troubleshooting")
print()

# ============================================================
# RESUMO FINAL
# ============================================================

print("="*70)
print("🎉 ESTRUTURA DO UNITY CATALOG CRIADA COM SUCESSO!")
print("="*70)
print()
print(f"📍 Catálogo: {catalog}")
print()
print("📂 Schemas criados:")
print(f"   1. {catalog}.{schema_bronze}  → Camada Bronze (Raw)")
print(f"   2. {catalog}.{schema_silver}  → Camada Silver (Conform)")
print(f"   3. {catalog}.{schema_gold}    → Camada Gold (Marts)")
print(f"   4. {catalog}.{schema_ops}     → Controle Operacional")
print(f"   5. {catalog}.{schema_qa}      → Qualidade de Dados")
print(f"   6. {catalog}.{schema_log}     → Logs Detalhados")
print()
print("✅ Pronto para criar as tabelas de controle!")
print("="*70)

## 🔧 Bloco 4: Tabelas de Controle Operacional (ops)

### 📋 Descrição

Este bloco cria as **4 tabelas de controle operacional** que garantem que o pipeline funcione de forma confiável, rastreável e com capacidade de replay.

---

### 📊 Tabela 1: `ops.ingestion_runs`

**Função:** Registra **cada execução completa** do pipeline.

**Quando é usada:**
* No **início** da execução: insere 1 linha com `status='RUNNING'`
* No **fim** da execução: atualiza a linha com `status='SUCCESS'` ou `'FAILED'`

**Colunas principais:**
* `load_id`: UUID único desta execução (chave primária)
* `start_ts`, `end_ts`: Horários de início e fim
* `status`: `RUNNING`, `SUCCESS`, `FAILED`
* `env`: Ambiente (dev/hml/prd)
* `run_date`: Data de referência
* `full_refresh`: Se foi recarga completa ou incremental
* `records_processed`: Total de registros processados
* `errors_count`: Quantidade de erros

**Por que é importante:**
* Histórico completo de execuções
* Troubleshooting: "Quando foi a última execução com sucesso?"
* Métricas: volume processado, taxa de erro, tempo de execução

---

### 📊 Tabela 2: `ops.ingestion_requests`

**Função:** Registra **cada requisição HTTP individual** feita à API.

**Quando é usada:**
* Após cada chamada HTTP à API da Câmara
* Exemplo: ao buscar página 1 de deputados, grava 1 linha

**Colunas principais:**
* `request_id`: UUID único da requisição
* `load_id`: FK para `ops.ingestion_runs` (qual execução fez esta request)
* `source_endpoint`: Endpoint chamado (ex: `/deputados`)
* `request_params`: Parâmetros JSON (paginação, filtros)
* `http_status`: Código HTTP (200=sucesso, 400/500=erro)
* `response_time_ms`: Tempo de resposta em milissegundos
* `records_returned`: Quantos registros a API retornou
* `error_message`: Mensagem de erro (se houver)

**Por que é importante:**
* Monitoramento: "API está lenta?", "Taxa de erro aumentou?"
* Troubleshooting: "Qual request específico falhou?"
* Auditoria: prova de que chamamos a API e quando

---

### 📊 Tabela 3: `ops.checkpoints`

**Função:** Controla **ingestão incremental** por endpoint.

**Como funciona:**
* Armazena o **último valor processado** para cada endpoint
* Exemplo: endpoint `/deputados` → último `deputado_id` processado = `12345`
* Na próxima execução incremental, busca apenas dados **após** `12345`

**Colunas principais:**
* `source_endpoint`: Endpoint (chave, ex: `/deputados`)
* `checkpoint_type`: Tipo do checkpoint (`id`, `date`, `page`)
* `checkpoint_value`: Último valor processado (string genérica)
* `last_load_id`: Qual execução atualizou este checkpoint
* `updated_at`: Quando foi atualizado

**Exemplos reais:**

| source_endpoint | checkpoint_type | checkpoint_value | Significado |
|----------------|----------------|-----------------|-------------|
| `/deputados` | `id` | `204560` | Último deputado_id processado |
| `/votacoes` | `date` | `2026-04-27` | Última data processada |
| `/ceap/{id}/despesas` | `page` | `15` | Última página processada |

**Por que é importante:**
* Evita reprocessar os mesmos dados toda vez (economia de tempo/custo)
* Permite replay: deletar checkpoint → reprocessa tudo
* Garante idempotência: mesma execução = mesmo resultado

---

### 📊 Tabela 4: `ops.dead_letter`

**Função:** Armazena **payloads que falharam** validação básica.

**Quando é usada:**
* Quando um payload JSON vem mal-formado da API
* Quando validação de schema falha
* Quando conversão de tipo falha

**Colunas principais:**
* `payload`: JSON raw que falhou (STRING)
* `source_endpoint`: De onde veio
* `error_message`: Por que falhou
* `error_type`: Categoria (`validation`, `parse`, `schema`)
* `load_id`: Qual execução detectou o erro

**Por que é importante:**
* **Resiliência**: pipeline não para por 1 registro ruim
* **Visibilidade**: sabemos exatamente quais dados falharam
* **Recuperação**: podemos reprocessar depois de corrigir

---

### 💡 Para Leigos

Imagine que você está gerenciando uma linha de produção de carros:

* **`ingestion_runs`** = Livro de ponto (registra cada turno de trabalho)
* **`ingestion_requests`** = Log de cada peça recebida do fornecedor
* **`checkpoints`** = Marcador de onde você parou na linha de montagem
* **`dead_letter`** = Caixa de "peças defeituosas" para análise posterior

Sem essas 4 tabelas, você não consegue:
* Saber se o trabalho foi concluído
* Continuar de onde parou
* Identificar problemas
* Recuperar de falhas

In [0]:
# ============================================================
# TABELA OPS 1: ingestion_runs
# ============================================================
# Registra cada execução completa do pipeline.
# 1 linha por execução (identificada pelo load_id).
# ============================================================

print("📊 Criando tabela ops.ingestion_runs...")

# Nome completo da tabela
tabela_runs = f"{catalog}.{schema_ops}.ingestion_runs"

# SQL para criar a tabela
sql_runs = f"""
CREATE TABLE IF NOT EXISTS {tabela_runs} (
    -- Identificação da execução
    load_id STRING NOT NULL COMMENT 'UUID único desta execução (chave primária)',
    
    -- Tempo de execução
    start_ts TIMESTAMP NOT NULL COMMENT 'Horário de início da execução',
    end_ts TIMESTAMP COMMENT 'Horário de fim da execução (NULL se ainda RUNNING)',
    execution_time_sec DOUBLE COMMENT 'Duração em segundos',
    
    -- Status
    status STRING NOT NULL COMMENT 'Status: RUNNING, SUCCESS, FAILED',
    
    -- Parâmetros de execução
    env STRING NOT NULL COMMENT 'Ambiente: dev, hml, prd',
    run_date DATE NOT NULL COMMENT 'Data de referência desta execução',
    full_refresh BOOLEAN NOT NULL COMMENT 'Se foi recarga completa (true) ou incremental (false)',
    
    -- Métricas agregadas
    total_entities_processed INT COMMENT 'Quantidade de entidades processadas',
    total_records_ingested LONG COMMENT 'Total de registros ingeridos',
    total_api_calls INT COMMENT 'Total de chamadas HTTP feitas',
    
    -- Informações de pipeline
    pipeline_version STRING COMMENT 'Versão do código do pipeline',
    triggered_by STRING COMMENT 'Como foi iniciado: manual, schedule, api',
    
    -- Erros
    error_message STRING COMMENT 'Mensagem de erro se status=FAILED',
    error_type STRING COMMENT 'Tipo do erro: api, validation, processing'
)
USING DELTA
COMMENT 'Histórico de execuções do pipeline (1 linha por run)'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
)
"""

# Executa a criação
spark.sql(sql_runs)

print(f"✅ Tabela {tabela_runs} criada/validada")
print(f"   → 1 linha por execução do pipeline")
print(f"   → Colunas principais: load_id, start_ts, end_ts, status, env, run_date")
print()

In [0]:
# ============================================================
# TABELA OPS 2: ingestion_requests
# ============================================================
# Registra cada requisição HTTP individual à API.
# Múltiplas linhas por execução (1 por chamada HTTP).
# ============================================================

print("📊 Criando tabela ops.ingestion_requests...")

tabela_requests = f"{catalog}.{schema_ops}.ingestion_requests"

sql_requests = f"""
CREATE TABLE IF NOT EXISTS {tabela_requests} (
    -- Identificação
    request_id STRING NOT NULL COMMENT 'UUID único desta requisição HTTP',
    load_id STRING NOT NULL COMMENT 'FK para ops.ingestion_runs (qual execução)',
    
    -- Detalhes da requisição
    endpoint STRING NOT NULL COMMENT 'Endpoint da API chamado (ex: /deputados)',
    http_method STRING COMMENT 'Método HTTP: GET, POST, etc',
    request_params STRING COMMENT 'Parâmetros da requisição em formato JSON',
    
    -- Resposta HTTP
    http_status INT COMMENT 'Código HTTP de resposta (200, 400, 500, etc)',
    response_time_ms DOUBLE COMMENT 'Tempo de resposta em milissegundos',
    response_body STRING COMMENT 'Corpo da resposta (se necessário armazenar)',
    
    -- Timestamp
    request_ts TIMESTAMP COMMENT 'Momento exato da requisição',
    
    -- Erros
    error_message STRING COMMENT 'Mensagem de erro (se http_status != 200)',
    retry_count INT COMMENT 'Quantas tentativas foram feitas'
)
USING DELTA
COMMENT 'Log de requisições HTTP à API (1 linha por request)'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
)
"""

spark.sql(sql_requests)

print(f"✅ Tabela {tabela_requests} criada/validada")
print(f"   → 1 linha por requisição HTTP à API")
print(f"   → Permite monitorar performance e erros da API")
print()

In [0]:
# ============================================================
# TABELA OPS 3: checkpoints
# ============================================================
# Controla ingestão incremental por endpoint.
# Armazena o último valor processado para cada fonte.
# ============================================================

print("📊 Criando tabela ops.checkpoints...")

tabela_checkpoints = f"{catalog}.{schema_ops}.checkpoints"

sql_checkpoints = f"""
CREATE TABLE IF NOT EXISTS {tabela_checkpoints} (
    -- Identificação da fonte
    source_endpoint STRING NOT NULL COMMENT 'Endpoint da API (ex: /deputados, /votacoes)',
    
    -- Tipo e valor do checkpoint
    checkpoint_type STRING NOT NULL COMMENT 'Tipo: max_id, last_date, last_page, offset',
    checkpoint_value STRING NOT NULL COMMENT 'Último valor processado (string genérica)',
    
    -- Rastreabilidade
    updated_ts TIMESTAMP COMMENT 'Última atualização',
    load_id STRING COMMENT 'Load ID da última execução que atualizou'
)
USING DELTA
COMMENT 'Controle de ingestão incremental (watermarks por endpoint)'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
)
"""

spark.sql(sql_checkpoints)

print(f"✅ Tabela {tabela_checkpoints} criada/validada")
print(f"   → 1 linha por endpoint (atualizada a cada execução incremental)")
print(f"   → Permite replay: deletar checkpoint = reprocessar tudo")
print()

# Exemplo de como funciona
print("📝 Exemplo de checkpoint:")
print("   source_endpoint='deputados', checkpoint_type='max_id', checkpoint_value='204560'")
print("   → Significa: último deputado_id processado foi 204560")
print("   → Próxima execução busca apenas IDs > 204560")
print()

In [0]:
# ============================================================
# TABELA OPS 4: dead_letter
# ============================================================
# Armazena payloads que falharam validação básica.
# Garante que nenhum dado é perdido mesmo em caso de erro.
# ============================================================

print("📊 Criando tabela ops.dead_letter...")

tabela_dead_letter = f"{catalog}.{schema_ops}.dead_letter"

sql_dead_letter = f"""
CREATE TABLE IF NOT EXISTS {tabela_dead_letter} (
    -- Identificação
    dead_letter_id STRING NOT NULL COMMENT 'UUID único deste registro',
    load_id STRING NOT NULL COMMENT 'Load ID da execução que detectou o erro',
    
    -- Payload problemático
    payload STRING NOT NULL COMMENT 'Payload JSON raw que falhou',
    record_hash STRING COMMENT 'Hash SHA256 do payload (para deduplicação)',
    
    -- Origem
    source_endpoint STRING NOT NULL COMMENT 'Endpoint de origem (ex: /deputados)',
    source_request_id STRING COMMENT 'FK para ops.ingestion_requests (qual request)',
    
    -- Erro
    error_message STRING NOT NULL COMMENT 'Mensagem de erro detalhada',
    error_type STRING NOT NULL COMMENT 'Tipo: validation, parse, schema, business_rule',
    error_stack_trace STRING COMMENT 'Stack trace completo do erro (opcional)',
    
    -- Timestamps
    ingestion_ts TIMESTAMP COMMENT 'Quando tentamos processar',
    created_ts TIMESTAMP COMMENT 'Quando foi registrado aqui'
)
USING DELTA
COMMENT 'Payloads que falharam validação (Dead Letter Queue)'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
)
"""

spark.sql(sql_dead_letter)

print(f"✅ Tabela {tabela_dead_letter} criada/validada")
print(f"   → Armazena registros problemáticos sem quebrar o pipeline")
print(f"   → Permite reprocessamento após correção")
print()

print("="*70)
print("🎉 4 TABELAS OPS CRIADAS COM SUCESSO!")
print("="*70)
print()
print("📊 Tabelas operacionais disponíveis:")
print(f"   1. {catalog}.{schema_ops}.ingestion_runs      → Histórico de execuções")
print(f"   2. {catalog}.{schema_ops}.ingestion_requests  → Log de chamadas HTTP")
print(f"   3. {catalog}.{schema_ops}.checkpoints         → Controle incremental")
print(f"   4. {catalog}.{schema_ops}.dead_letter         → Payloads problemáticos")
print()
print("✅ Controle operacional pronto!")
print("="*70)

In [0]:
# ============================================================
# [TEMPORÁRIO] CRIAR TODAS AS TABELAS LOG E QA DE UMA VEZ
# ============================================================
# Esta célula cria as 7 tabelas (4 LOG + 3 QA) rapidamente
# sem DEFAULT para evitar erros.
# ============================================================

print("🚀 CRIAÇÃO EXPRESSA: 7 tabelas (LOG + QA)")
print("=" * 70)
print()

# ===== TABELAS LOG =====
print("📊 Criando 4 tabelas LOG...")

# LOG 1
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema_log}.bronze_ingest_runs (
    run_id STRING NOT NULL, load_id STRING NOT NULL, entity_name STRING NOT NULL,
    source_endpoint STRING NOT NULL, source_type STRING, start_ts TIMESTAMP NOT NULL,
    end_ts TIMESTAMP, execution_time_sec DOUBLE, status STRING NOT NULL,
    records_ingested LONG, records_failed INT, api_calls_count INT,
    avg_response_time_ms DOUBLE, total_pages INT, checkpoint_before STRING,
    checkpoint_after STRING, error_message STRING, error_type STRING
) USING DELTA TBLPROPERTIES ('delta.autoOptimize.optimizeWrite'='true')
""")
print("✅ log.bronze_ingest_runs")

# LOG 2
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema_log}.silver_transform_runs (
    run_id STRING NOT NULL, load_id STRING NOT NULL, entity_name STRING NOT NULL,
    source_table STRING NOT NULL, target_table STRING NOT NULL, start_ts TIMESTAMP NOT NULL,
    end_ts TIMESTAMP, execution_time_sec DOUBLE, status STRING NOT NULL,
    transformation_type STRING, records_read LONG, records_written LONG,
    records_rejected LONG, error_message STRING
) USING DELTA TBLPROPERTIES ('delta.autoOptimize.optimizeWrite'='true')
""")
print("✅ log.silver_transform_runs")

# LOG 3
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema_log}.gold_mart_runs (
    run_id STRING NOT NULL, load_id STRING NOT NULL, mart_name STRING NOT NULL,
    start_ts TIMESTAMP NOT NULL, end_ts TIMESTAMP, execution_time_sec DOUBLE,
    status STRING NOT NULL, mart_type STRING, records_written LONG,
    records_updated LONG, dependencies STRING, joins_count INT, error_message STRING
) USING DELTA TBLPROPERTIES ('delta.autoOptimize.optimizeWrite'='true')
""")
print("✅ log.gold_mart_runs")

# LOG 4
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema_log}.quality_checks (
    check_id STRING NOT NULL, load_id STRING NOT NULL, layer STRING NOT NULL,
    table_name STRING NOT NULL, entity_name STRING, rule_id STRING NOT NULL,
    rule_name STRING, rule_type STRING NOT NULL, rule_category STRING,
    status STRING NOT NULL, observed_value STRING, expected_value STRING,
    deviation_value DOUBLE, execution_time_sec DOUBLE, error_message STRING,
    check_ts TIMESTAMP
) USING DELTA PARTITIONED BY (layer, rule_type)
TBLPROPERTIES ('delta.autoOptimize.optimizeWrite'='true')
""")
print("✅ log.quality_checks")

print()
print("🔍 Criando 3 tabelas QA...")

# QA 1
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema_qa}.quality_results (
    result_id STRING NOT NULL, load_id STRING NOT NULL, layer STRING NOT NULL,
    table_name STRING NOT NULL, entity_name STRING, check_ts TIMESTAMP NOT NULL,
    qa_passed BOOLEAN NOT NULL, total_rules INT NOT NULL, rules_passed INT,
    rules_failed INT, hard_rules_count INT, hard_passed INT, hard_failed INT,
    soft_rules_count INT, soft_passed INT, soft_failed INT, total_records LONG,
    valid_records LONG, rejected_records LONG, reject_percentage DOUBLE,
    failed_rules_summary STRING, execution_time_sec DOUBLE
) USING DELTA PARTITIONED BY (layer)
TBLPROPERTIES ('delta.autoOptimize.optimizeWrite'='true')
""")
print("✅ qa.quality_results")

# QA 2
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema_qa}.quality_metrics (
    metric_id STRING NOT NULL, load_id STRING NOT NULL, layer STRING NOT NULL,
    table_name STRING NOT NULL, entity_name STRING, column_name STRING,
    metric_ts TIMESTAMP NOT NULL, metric_type STRING NOT NULL, metric_name STRING NOT NULL,
    metric_value DOUBLE, metric_value_str STRING, null_count LONG, null_percentage DOUBLE,
    distinct_count LONG, duplicate_count LONG, min_value STRING, max_value STRING,
    avg_value DOUBLE, stddev_value DOUBLE, threshold_min DOUBLE, threshold_max DOUBLE,
    threshold_violated BOOLEAN, sample_values STRING, notes STRING
) USING DELTA PARTITIONED BY (layer, metric_type)
TBLPROPERTIES ('delta.autoOptimize.optimizeWrite'='true')
""")
print("✅ qa.quality_metrics")

# QA 3
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema_qa}.rejects (
    reject_id STRING NOT NULL, load_id STRING NOT NULL, source_table STRING NOT NULL,
    target_table STRING NOT NULL, entity_name STRING NOT NULL, reject_ts TIMESTAMP NOT NULL,
    reject_reason STRING NOT NULL, reject_type STRING, rule_id STRING, record_key STRING,
    record_payload STRING, field_name STRING, field_value STRING, expected_value STRING,
    severity STRING NOT NULL, action_taken STRING, error_message STRING, stack_trace STRING,
    can_be_replayed BOOLEAN, replay_count INT
) USING DELTA PARTITIONED BY (entity_name)
TBLPROPERTIES ('delta.autoOptimize.optimizeWrite'='true')
""")
print("✅ qa.rejects")

print()
print("=" * 70)
print("🎉 7 TABELAS CRIADAS: 4 LOG + 3 QA")
print("=" * 70)
print()

## 📊 Bloco 5: Tabelas de Log Detalhado (log)

### 📋 Descrição

Este bloco cria **6 tabelas de log granular** para troubleshooting detalhado por camada do pipeline.

**Diferença entre `ops` e `log`:**

* **`ops`**: Logs **operacionais essenciais** (execuções, checkpoints, dead letter)
* **`log`**: Logs **detalhados por camada** para diagnóstico profundo

**Quando usar `log`:**
* "Qual entidade Bronze falhou?"
* "Quanto tempo levou cada transformação Silver?"
* "Quantos registros foram rejeitados na Silver?"
* "Qual mart Gold está demorando mais?"

---

### 📊 Tabela 1: `log.bronze_ingest_runs`

**Função:** Log de cada **ingestão Bronze por entidade**.

**Exemplo:** Se em 1 execução processamos deputados + frentes + eventos, terá 3 linhas aqui.

**Colunas principais:**
* `entity_name`: Nome da entidade (deputados, frentes, ceap, etc)
* `source_endpoint`: Endpoint da API
* `records_ingested`: Quantos registros foram gravados
* `api_calls_count`: Quantas chamadas HTTP foram feitas
* `checkpoint_before`, `checkpoint_after`: Estado do checkpoint (debug incremental)

---

### 📊 Tabela 2: `log.bronze_api_requests`

**Função:** Espelho de `ops.ingestion_requests` mas com **mais detalhes**.

**Diferença:**
* `ops.ingestion_requests`: Log mínimo necessário
* `log.bronze_api_requests`: Inclui request/response headers, retry attempts, etc

---

### 📊 Tabela 3: `log.silver_transform_runs`

**Função:** Log de cada **transformação Silver por tabela**.

**Exemplo:** Transformar `bronze.deputados_raw` → `silver.deputados` = 1 linha aqui.

**Colunas principais:**
* `entity_name`: Nome da entidade
* `source_table`: Tabela Bronze de origem
* `target_table`: Tabela Silver destino
* `records_read`: Quantos registros leu da Bronze
* `records_written`: Quantos gravou na Silver
* `records_rejected`: Quantos falharam validação
* `transformation_type`: parse, dedupe, scd2, etc

---

### 📊 Tabela 4: `log.gold_mart_runs`

**Função:** Log de cada **criação de mart Gold**.

**Colunas principais:**
* `mart_name`: Nome do mart (ex: mart_frentes, mart_ceap_anomalias)
* `dependencies`: Quais tabelas Silver foram usadas
* `records_written`: Quantos registros no mart
* `execution_time_sec`: Tempo de execução

---

### 📊 Tabela 5: `log.quality_checks`

**Função:** Histórico **completo** de todas as regras QA executadas.

**Diferença de `qa.quality_results`:**
* `qa.quality_results`: Resultados **consolidados** (1 linha por regra por execução)
* `log.quality_checks`: Histórico **granular** com cada checagem individual

---

### 💡 Para Leigos

Se `ops` é o "livro de ponto" da fábrica, `log` é o "diário de bordo" com anotações detalhadas:

* **`ops`**: "Turno iniciou às 8h, terminou às 17h, produziu 1000 peças"
* **`log`**: "Máquina A produziu 300 peças em 2h, máquina B produziu 700 em 3h, 5 peças rejeitadas por defeito X"

Quando algo der errado, `log` tem todos os detalhes para investigar!

In [0]:
# ============================================================
# TABELA LOG 1: bronze_ingest_runs
# ============================================================
# Log de cada ingestão Bronze por entidade.
# Granularidade: 1 linha por entidade por execução.
# ============================================================

print("📊 Criando tabela log.bronze_ingest_runs...")

tabela_bronze_ingest = f"{catalog}.{schema_log}.bronze_ingest_runs"

sql_bronze_ingest = f"""
CREATE TABLE IF NOT EXISTS {tabela_bronze_ingest} (
    -- Identificação
    run_id STRING NOT NULL COMMENT 'UUID único desta ingestão',
    load_id STRING NOT NULL COMMENT 'FK para ops.ingestion_runs',
    entity_name STRING NOT NULL COMMENT 'Nome da entidade (deputados, frentes, ceap, etc)',
    
    -- Fonte
    source_endpoint STRING NOT NULL COMMENT 'Endpoint da API',
    source_type STRING COMMENT 'Tipo: api_rest, file, database, etc',
    
    -- Tempo de execução
    start_ts TIMESTAMP NOT NULL COMMENT 'Início da ingestão desta entidade',
    end_ts TIMESTAMP COMMENT 'Fim da ingestão',
    execution_time_sec DOUBLE COMMENT 'Tempo total em segundos',
    
    -- Status e resultados
    status STRING NOT NULL COMMENT 'Status: RUNNING, SUCCESS, FAILED, PARTIAL',
    records_ingested LONG COMMENT 'Quantidade de registros gravados',
    records_failed INT COMMENT 'Quantidade que foi para dead_letter',
    
    -- Métricas de API
    api_calls_count INT COMMENT 'Quantas requisições HTTP foram feitas',
    avg_response_time_ms DOUBLE COMMENT 'Tempo médio de resposta da API (ms)',
    total_pages INT COMMENT 'Total de páginas processadas (se paginado)',
    
    -- Controle incremental
    checkpoint_before STRING COMMENT 'Valor do checkpoint antes da execução',
    checkpoint_after STRING COMMENT 'Valor do checkpoint depois da execução',
    is_full_refresh BOOLEAN COMMENT 'Se foi recarga completa (ignora checkpoint)',
    
    -- Erros
    error_message STRING COMMENT 'Mensagem de erro (se status=FAILED)',
    error_count INT COMMENT 'Quantidade de erros durante ingestão',
    
    -- Metadados
    target_table STRING COMMENT 'Nome da tabela Bronze gravada',
    created_ts TIMESTAMP DEFAULT current_timestamp()
)
USING DELTA
COMMENT 'Log detalhado de ingestões Bronze por entidade'
"""

spark.sql(sql_bronze_ingest)

print(f"✅ Tabela {tabela_bronze_ingest} criada/validada")
print(f"   → 1 linha por entidade por execução")
print(f"   → Monitora performance e sucesso de cada ingestão")
print()

In [0]:
# ============================================================
# TABELA LOG 2: silver_transform_runs
# ============================================================
# Log de cada transformação Silver por tabela.
# Granularidade: 1 linha por tabela Silver por execução.
# ============================================================

print("📊 Criando tabela log.silver_transform_runs...")

tabela_silver_transform = f"{catalog}.{schema_log}.silver_transform_runs"

sql_silver_transform = f"""
CREATE TABLE IF NOT EXISTS {tabela_silver_transform} (
    -- Identificação
    run_id STRING NOT NULL COMMENT 'UUID único desta transformação',
    load_id STRING NOT NULL COMMENT 'FK para ops.ingestion_runs',
    entity_name STRING NOT NULL COMMENT 'Nome da entidade',
    
    -- Origem e destino
    source_table STRING NOT NULL COMMENT 'Tabela Bronze de origem (catalog.schema.table)',
    target_table STRING NOT NULL COMMENT 'Tabela Silver destino (catalog.schema.table)',
    
    -- Tempo de execução
    start_ts TIMESTAMP NOT NULL COMMENT 'Início da transformação',
    end_ts TIMESTAMP COMMENT 'Fim da transformação',
    execution_time_sec DOUBLE COMMENT 'Tempo total em segundos',
    
    -- Status e resultados
    status STRING NOT NULL COMMENT 'Status: RUNNING, SUCCESS, FAILED',
    transformation_type STRING COMMENT 'Tipo: parse, dedupe, scd2, merge, etc',
    
    -- Métricas de dados
    records_read LONG COMMENT 'Registros lidos da Bronze',
    records_written LONG COMMENT 'Registros gravados na Silver',
    records_rejected LONG COMMENT 'Registros rejeitados (foram para qa.rejects)',
    records_updated LONG COMMENT 'Registros atualizados (em caso de merge/SCD2)',
    records_inserted LONG COMMENT 'Registros novos inseridos',
    records_deleted LONG COMMENT 'Registros deletados (soft delete)',
    
    -- Qualidade
    duplicate_records INT COMMENT 'Duplicatas encontradas e removidas',
    null_key_records INT COMMENT 'Registros com chave primária nula',
    data_quality_score DOUBLE COMMENT 'Score de qualidade (0-100)',
    
    -- Erros
    error_message STRING COMMENT 'Mensagem de erro (se status=FAILED)',
    error_sample STRING COMMENT 'Exemplo de registro que falhou',
    
    -- Metadados
    partition_columns STRING COMMENT 'Colunas de particionamento',
    created_ts TIMESTAMP DEFAULT current_timestamp()
)
USING DELTA
COMMENT 'Log detalhado de transformações Silver'
"""

spark.sql(sql_silver_transform)

print(f"✅ Tabela {tabela_silver_transform} criada/validada")
print(f"   → 1 linha por transformação Silver")
print(f"   → Rastreia parse, dedupe, SCD2, etc")
print()

In [0]:
# ============================================================
# TABELA LOG 3: gold_mart_runs
# ============================================================
# Log de cada criação de mart Gold.
# Granularidade: 1 linha por mart por execução.
# ============================================================

print("📊 Criando tabela log.gold_mart_runs...")

tabela_gold_mart = f"{catalog}.{schema_log}.gold_mart_runs"

sql_gold_mart = f"""
CREATE TABLE IF NOT EXISTS {tabela_gold_mart} (
    -- Identificação
    run_id STRING NOT NULL COMMENT 'UUID único desta criação de mart',
    load_id STRING NOT NULL COMMENT 'FK para ops.ingestion_runs',
    mart_name STRING NOT NULL COMMENT 'Nome do mart (ex: mart_frentes, mart_ceap)',
    
    -- Tempo de execução
    start_ts TIMESTAMP NOT NULL COMMENT 'Início da criação do mart',
    end_ts TIMESTAMP COMMENT 'Fim da criação',
    execution_time_sec DOUBLE COMMENT 'Tempo total em segundos',
    
    -- Status e resultados
    status STRING NOT NULL COMMENT 'Status: RUNNING, SUCCESS, FAILED',
    mart_type STRING COMMENT 'Tipo: fact, dimension, aggregate, report, etc',
    
    -- Métricas
    records_written LONG COMMENT 'Quantidade de registros no mart',
    records_updated LONG COMMENT 'Registros atualizados (se incremental)',
    
    -- Dependências
    dependencies STRING COMMENT 'Tabelas Silver usadas (JSON array)',
    joins_count INT COMMENT 'Quantidade de joins realizados',
    aggregations STRING COMMENT 'Agregações realizadas (JSON)',
    
    -- Performance
    shuffle_read_mb DOUBLE COMMENT 'Dados lidos via shuffle (MB)',
    shuffle_write_mb DOUBLE COMMENT 'Dados escritos via shuffle (MB)',
    cpu_time_sec DOUBLE COMMENT 'Tempo de CPU usado',
    
    -- Erros
    error_message STRING COMMENT 'Mensagem de erro (se status=FAILED)',
    
    -- Metadados
    target_table STRING COMMENT 'Nome completo da tabela Gold',
    refresh_mode STRING COMMENT 'full, incremental, append',
    created_ts TIMESTAMP DEFAULT current_timestamp()
)
USING DELTA
COMMENT 'Log detalhado de criação de marts Gold'
"""

spark.sql(sql_gold_mart)

print(f"✅ Tabela {tabela_gold_mart} criada/validada")
print(f"   → 1 linha por mart Gold")
print(f"   → Monitora performance e dependências")
print()

In [0]:
# ============================================================
# TABELA LOG 4: quality_checks
# ============================================================
# Histórico completo de todas as regras QA executadas.
# Granularidade: 1 linha por regra por tabela por execução.
# ============================================================

print("📊 Criando tabela log.quality_checks...")

tabela_quality_checks = f"{catalog}.{schema_log}.quality_checks"

sql_quality_checks = f"""
CREATE TABLE IF NOT EXISTS {tabela_quality_checks} (
    -- Identificação
    check_id STRING NOT NULL COMMENT 'UUID único desta checagem',
    load_id STRING NOT NULL COMMENT 'FK para ops.ingestion_runs',
    
    -- Contexto
    layer STRING NOT NULL COMMENT 'Camada: bronze, silver, gold',
    table_name STRING NOT NULL COMMENT 'Nome completo da tabela checada',
    entity_name STRING COMMENT 'Nome da entidade (opcional)',
    
    -- Regra QA
    rule_id STRING NOT NULL COMMENT 'ID da regra (ex: PK_NOT_NULL, NO_DUPLICATES)',
    rule_name STRING COMMENT 'Nome descritivo da regra',
    rule_type STRING NOT NULL COMMENT 'Tipo: HARD (bloqueante) ou SOFT (warning)',
    rule_category STRING COMMENT 'Categoria: completeness, uniqueness, validity, etc',
    
    -- Resultado
    status STRING NOT NULL COMMENT 'Resultado: PASS, FAIL, WARNING, SKIP',
    observed_value STRING COMMENT 'Valor observado (ex: "5 duplicatas encontradas")',
    expected_value STRING COMMENT 'Valor esperado (ex: "0 duplicatas")',
    threshold STRING COMMENT 'Limite aceitável (ex: "< 1%")',
    
    -- Detalhes
    sample_query STRING COMMENT 'Query SQL para reproduzir/investigar',
    sample_records STRING COMMENT 'Exemplos de registros problemáticos (JSON)',
    affected_records_count LONG COMMENT 'Quantidade de registros afetados',
    
    -- Impacto
    severity STRING COMMENT 'Severidade: CRITICAL, HIGH, MEDIUM, LOW',
    action_taken STRING COMMENT 'Ação tomada: BLOCK, REJECT, LOG, NONE',
    
    -- Timestamps
    executed_at TIMESTAMP DEFAULT current_timestamp() COMMENT 'Quando foi executada',
    execution_time_ms LONG COMMENT 'Tempo de execução em ms',
    
    -- Metadados
    created_ts TIMESTAMP DEFAULT current_timestamp()
)
USING DELTA
COMMENT 'Histórico granular de checagens de qualidade'
"""

spark.sql(sql_quality_checks)

print(f"✅ Tabela {tabela_quality_checks} criada/validada")
print(f"   → Histórico completo de regras QA")
print(f"   → HARD rules bloqueiam pipeline, SOFT rules apenas alertam")
print()

print("="*70)
print("🎉 4 TABELAS LOG CRIADAS COM SUCESSO!")
print("="*70)
print()
print("📊 Tabelas de log detalhado disponíveis:")
print(f"   1. {catalog}.{schema_log}.bronze_ingest_runs     → Ingestões Bronze")
print(f"   2. {catalog}.{schema_log}.silver_transform_runs  → Transformações Silver")
print(f"   3. {catalog}.{schema_log}.gold_mart_runs         → Marts Gold")
print(f"   4. {catalog}.{schema_log}.quality_checks         → Histórico QA")
print()
print("✅ Logs detalhados prontos para troubleshooting!")
print("="*70)

## 🔍 Bloco 6: Tabelas de Qualidade (qa)

### 📋 Descrição

Este bloco cria **3 tabelas de controle de qualidade** para garantir a confiabilidade dos dados.

**Filosofia de Qualidade no Pipeline:**

* **Regras HARD**: Bloqueantes — falha interrompe a execução
* **Regras SOFT**: Warnings — registra mas continua processando

**Exemplo de uso:**
* HARD: "PK não pode ser nula" → bloqueia se falhar
* SOFT: "Menos de 5% de nulos" → alerta mas continua

---

### 🔍 Tabela 1: `qa.quality_results`

**Função:** Resultados **consolidados** de todas as regras QA por execução.

**Granularidade:** 1 linha por tabela por execução.

**Uso típico:**
```sql
-- Ver status de qualidade da última execução
SELECT table_name, qa_passed, total_rules, hard_fails, soft_fails
FROM qa.quality_results
WHERE load_id = '<ultimo_load_id>'
```

---

### 🔍 Tabela 2: `qa.quality_metrics`

**Função:** Métricas **agregadas** de qualidade ao longo do tempo.

**Uso típico:**
* Calcular porcentagem de nulos por coluna
* Rastrear tendências de qualidade
* Identificar degradação de dados

**Exemplo:**
```sql
-- Colunas com mais de 10% de nulos
SELECT table_name, column_name, null_percentage
FROM qa.quality_metrics
WHERE null_percentage > 0.10
ORDER BY null_percentage DESC
```

---

### 🔍 Tabela 3: `qa.rejects`

**Função:** Registros **rejeitados** durante as transformações Silver.

**Diferença com `ops.dead_letter`:**
* **`dead_letter`**: Payloads JSON que falharam no parse (Bronze)
* **`rejects`**: Registros que falharam validação de negócio (Silver)

**Exemplo:**
```sql
-- Ver rejeitados da Silver com motivo
SELECT entity_name, reject_reason, COUNT(*) as qty
FROM qa.rejects
WHERE load_id = '<ultimo_load_id>'
GROUP BY entity_name, reject_reason
```

In [0]:
# ============================================================
# TABELA QA 1: quality_results
# ============================================================
# Resultados consolidados de qualidade por tabela.
# Granularidade: 1 linha por tabela por execução.
# ============================================================

print("🔍 Criando tabela qa.quality_results...")

tabela_quality_results = f"{catalog}.{schema_qa}.quality_results"

sql_quality_results = f"""
CREATE TABLE IF NOT EXISTS {tabela_quality_results} (
    -- Identificação
    result_id STRING NOT NULL COMMENT 'UUID único deste resultado consolidado',
    load_id STRING NOT NULL COMMENT 'FK para ops.ingestion_runs',
    
    -- Contexto
    layer STRING NOT NULL COMMENT 'Camada: bronze, silver, gold',
    table_name STRING NOT NULL COMMENT 'Nome completo da tabela (catalog.schema.table)',
    entity_name STRING COMMENT 'Nome da entidade',
    
    -- Timestamp
    check_ts TIMESTAMP NOT NULL COMMENT 'Timestamp da checagem de qualidade',
    
    -- Resultado agregado
    qa_passed BOOLEAN NOT NULL COMMENT 'TRUE se TODAS as regras HARD passaram',
    total_rules INT NOT NULL COMMENT 'Total de regras executadas',
    rules_passed INT COMMENT 'Quantidade de regras que passaram',
    rules_failed INT COMMENT 'Quantidade de regras que falharam',
    
    -- Detalhamento por severidade
    hard_rules_count INT COMMENT 'Quantidade de regras HARD',
    hard_passed INT COMMENT 'Regras HARD que passaram',
    hard_failed INT COMMENT 'Regras HARD que falharam',
    
    soft_rules_count INT COMMENT 'Quantidade de regras SOFT',
    soft_passed INT COMMENT 'Regras SOFT que passaram',
    soft_failed INT COMMENT 'Regras SOFT que falharam (warnings)',
    
    -- Estatísticas de dados
    total_records LONG COMMENT 'Total de registros na tabela checada',
    valid_records LONG COMMENT 'Registros que passaram em todas as regras',
    rejected_records LONG COMMENT 'Registros rejeitados',
    reject_percentage DOUBLE COMMENT 'Porcentagem de rejeição',
    
    -- Sumário de regras que falharam
    failed_rules_summary STRING COMMENT 'Lista de rule_id que falharam (JSON array)',
    
    -- Metadados
    execution_time_sec DOUBLE COMMENT 'Tempo de execução das checagens (segundos)'
) 
COMMENT 'Resultados consolidados de qualidade por tabela'
PARTITIONED BY (layer)
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
)
"""

# Executar criação
spark.sql(sql_quality_results)

print(f"✅ Tabela {tabela_quality_results} criada com sucesso!")
print(f"   📊 Granularidade: 1 linha por tabela por execução")
print(f"   🎯 Uso: Monitorar status agregado de qualidade")
print()

In [0]:
# ============================================================
# TABELA QA 2: quality_metrics
# ============================================================
# Métricas agregadas de qualidade ao longo do tempo.
# Granularidade: 1 linha por coluna por métrica por execução.
# ============================================================

print("🔍 Criando tabela qa.quality_metrics...")

tabela_quality_metrics = f"{catalog}.{schema_qa}.quality_metrics"

sql_quality_metrics = f"""
CREATE TABLE IF NOT EXISTS {tabela_quality_metrics} (
    -- Identificação
    metric_id STRING NOT NULL COMMENT 'UUID único desta métrica',
    load_id STRING NOT NULL COMMENT 'FK para ops.ingestion_runs',
    
    -- Contexto
    layer STRING NOT NULL COMMENT 'Camada: bronze, silver, gold',
    table_name STRING NOT NULL COMMENT 'Nome completo da tabela',
    entity_name STRING COMMENT 'Nome da entidade',
    column_name STRING COMMENT 'Nome da coluna (NULL se métrica de tabela)',
    
    -- Timestamp
    metric_ts TIMESTAMP NOT NULL COMMENT 'Timestamp da coleta da métrica',
    
    -- Tipo de métrica
    metric_type STRING NOT NULL COMMENT 'Tipo: completeness, uniqueness, validity, accuracy, etc',
    metric_name STRING NOT NULL COMMENT 'Nome da métrica (ex: null_count, distinct_count)',
    
    -- Valores observados
    metric_value DOUBLE COMMENT 'Valor numérico da métrica',
    metric_value_str STRING COMMENT 'Valor string da métrica (se não numérico)',
    
    -- Estatísticas de completude
    null_count LONG COMMENT 'Quantidade de nulos',
    null_percentage DOUBLE COMMENT 'Porcentagem de nulos',
    
    -- Estatísticas de unicidade
    distinct_count LONG COMMENT 'Quantidade de valores distintos',
    duplicate_count LONG COMMENT 'Quantidade de duplicados',
    
    -- Estatísticas de distribuição
    min_value STRING COMMENT 'Valor mínimo (convertido para string)',
    max_value STRING COMMENT 'Valor máximo (convertido para string)',
    avg_value DOUBLE COMMENT 'Valor médio (se numérico)',
    stddev_value DOUBLE COMMENT 'Desvio padrão (se numérico)',
    
    -- Thresholds
    threshold_min DOUBLE COMMENT 'Limite mínimo esperado',
    threshold_max DOUBLE COMMENT 'Limite máximo esperado',
    threshold_violated BOOLEAN COMMENT 'TRUE se violou threshold',
    
    -- Contexto adicional
    sample_values STRING COMMENT 'Amostra de valores (JSON array)',
    notes STRING COMMENT 'Observações adicionais'
) 
COMMENT 'Métricas agregadas de qualidade ao longo do tempo'
PARTITIONED BY (layer, metric_type)
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
)
"""

# Executar criação
spark.sql(sql_quality_metrics)

print(f"✅ Tabela {tabela_quality_metrics} criada com sucesso!")
print(f"   📊 Granularidade: 1 linha por coluna por métrica por execução")
print(f"   🎯 Uso: Rastrear tendências de qualidade, identificar degradação")
print()

In [0]:
# ============================================================
# TABELA QA 3: rejects
# ============================================================
# Registros rejeitados durante transformações Silver.
# Granularidade: 1 linha por registro rejeitado.
# ============================================================

print("🔍 Criando tabela qa.rejects...")

tabela_rejects = f"{catalog}.{schema_qa}.rejects"

sql_rejects = f"""
CREATE TABLE IF NOT EXISTS {tabela_rejects} (
    -- Identificação
    reject_id STRING NOT NULL COMMENT 'UUID único desta rejeição',
    load_id STRING NOT NULL COMMENT 'FK para ops.ingestion_runs',
    
    -- Contexto
    source_table STRING NOT NULL COMMENT 'Tabela Bronze de origem',
    target_table STRING NOT NULL COMMENT 'Tabela Silver destino',
    entity_name STRING NOT NULL COMMENT 'Nome da entidade',
    
    -- Timestamp
    reject_ts TIMESTAMP NOT NULL COMMENT 'Timestamp da rejeição',
    
    -- Motivo da rejeição
    reject_reason STRING NOT NULL COMMENT 'Motivo da rejeição (ex: INVALID_DATE, MISSING_FK)',
    reject_type STRING COMMENT 'Tipo: VALIDATION_ERROR, BUSINESS_RULE, DATA_TYPE, etc',
    rule_id STRING COMMENT 'ID da regra QA que causou a rejeição',
    
    -- Dados do registro rejeitado
    record_key STRING COMMENT 'Chave do registro rejeitado (PK ou identificador único)',
    record_payload STRING COMMENT 'Payload completo do registro (JSON)',
    
    -- Detalhes do erro
    field_name STRING COMMENT 'Nome do campo que falhou validação',
    field_value STRING COMMENT 'Valor do campo que falhou',
    expected_value STRING COMMENT 'Valor esperado ou formato esperado',
    
    -- Severidade
    severity STRING NOT NULL COMMENT 'Severidade: CRITICAL, HIGH, MEDIUM, LOW',
    
    -- Ação tomada
    action_taken STRING COMMENT 'Ação: DROPPED, QUARANTINED, FIXED, LOGGED',
    
    -- Contexto adicional
    error_message STRING COMMENT 'Mensagem de erro detalhada',
    stack_trace STRING COMMENT 'Stack trace se houver exceção',
    
    -- Metadados
    can_be_replayed BOOLEAN COMMENT 'TRUE se o registro pode ser reprocessado',
    replay_count INT COMMENT 'Quantas vezes já foi reprocessado'
) 
COMMENT 'Registros rejeitados durante transformações Silver'
PARTITIONED BY (entity_name)
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
)
"""

# Executar criação
spark.sql(sql_rejects)

print(f"✅ Tabela {tabela_rejects} criada com sucesso!")
print(f"   📊 Granularidade: 1 linha por registro rejeitado")
print(f"   🎯 Uso: Investigar falhas de validação, replay de registros")
print()
print("🎉 BLOCO 6 COMPLETO: Tabelas QA criadas!")
print()

## 🆔 Bloco 7: Geração do Load ID

### 📋 Descrição

Este bloco gera o **Load ID** — o **identificador único** desta execução do pipeline.

**O que é o Load ID?**
* UUID hexadecimal de 32 caracteres
* Identificador **único** de cada execução do pipeline
* Usado como **Foreign Key** em TODAS as tabelas de log

**Por que é importante?**
* Permite rastrear **todos** os dados de uma execução
* Facilita debugging: "mostre tudo da execução X"
* Garante idempotência: mesma execução = mesmo load_id

**Onde é usado?**
```sql
-- Exemplo: Ver tudo de uma execução específica
SELECT * FROM ops.ingestion_runs WHERE load_id = '<load_id>'
SELECT * FROM log.bronze_ingest_runs WHERE load_id = '<load_id>'
SELECT * FROM qa.quality_results WHERE load_id = '<load_id>'
```

**Formato:**
```
Exemplo: a3f5b2c8d1e4f6a7b8c9d0e1f2a3b4c5
         └──────────────────────────────┘
              32 caracteres hexadecimais
```

---

### ⏰ Quando é gerado?

**Sempre no INÍCIO** da execução, logo após criar a infraestrutura.

**Atenção:** Em caso de retry, um **novo** load_id é gerado!

In [0]:
# ============================================================
# GERAÇÃO DO LOAD_ID
# ============================================================
# Gera identificador único (UUID) para esta execução.
# Será usado como FK em todas as tabelas de log.
# ============================================================

import uuid
from datetime import datetime

print("="*60)
print("🆔 GERANDO LOAD ID DA EXECUÇÃO")
print("="*60)
print()

# Gerar UUID único de 32 caracteres
load_id = uuid.uuid4().hex

# Capturar timestamp de início
start_ts = datetime.now()

print(f"✅ Load ID gerado com sucesso!")
print()
print(f"   🆔 Load ID: {load_id}")
print(f"   ⏰ Timestamp: {start_ts}")
print(f"   🌍 Ambiente: {env}")
print(f"   📅 Run Date: {run_date}")
print(f"   🔄 Full Refresh: {full_refresh}")
print()
print(f"📌 Este load_id será usado em TODAS as tabelas de log")
print(f"📌 Para rastrear esta execução, filtre: WHERE load_id = '{load_id}'")
print()
print("="*60)
print()

## 📝 Bloco 8: Registro Início da Execução

### 📋 Descrição

Este bloco registra o **início da execução** na tabela `ops.ingestion_runs`.

**O que faz?**
* Insere 1 linha em `ops.ingestion_runs` com `status = 'RUNNING'`
* Registra parâmetros da execução (env, run_date, full_refresh)
* Marca o timestamp de início

**Por que é importante?**
* Permite monitorar execuções **em andamento**
* Facilita identificação de execuções travadas
* Registra contexto da execução (ambiente, data de referência)

**Ciclo de vida do registro:**
```
1. INÍCIO: INSERT com status='RUNNING'
2. DURANTE: Pipeline processa dados
3. FIM: UPDATE com status='SUCCESS' ou 'FAILED' + end_ts
```

**Exemplo de consulta:**
```sql
-- Ver execuções em andamento
SELECT load_id, start_ts, env, run_date
FROM ops.ingestion_runs
WHERE status = 'RUNNING'
ORDER BY start_ts DESC
```

---

### ⚠️ Importante

* Este registro será **atualizado** no final da execução
* Se o pipeline falhar, o status ficará como `RUNNING`
* Para detectar falhas: buscar registros RUNNING com start_ts > 1h atrás

In [0]:
# ============================================================
# REGISTRO DE INÍCIO DA EXECUÇÃO
# ============================================================
# Insere registro em ops.ingestion_runs com status=RUNNING.
# Será atualizado no final com SUCCESS ou FAILED.
# ============================================================

print("="*60)
print("📝 REGISTRANDO INÍCIO DA EXECUÇÃO")
print("="*60)
print()

tabela_runs = f"{catalog}.{schema_ops}.ingestion_runs"

print(f"📁 Tabela: {tabela_runs}")
print(f"🆔 Load ID: {load_id}")
print()

# Inserir registro usando SQL direto
full_refresh_bool = 'true' if full_refresh == 'true' else 'false'

spark.sql(f"""
INSERT INTO {tabela_runs} (
    load_id, start_ts, end_ts, execution_time_sec, status,
    env, run_date, full_refresh, total_entities_processed,
    total_records_ingested, total_api_calls, pipeline_version,
    triggered_by, error_message, error_type
)
VALUES (
    '{load_id}',
    TIMESTAMP '{start_ts}',
    NULL,
    NULL,
    'RUNNING',
    '{env}',
    DATE '{run_date}',
    {full_refresh_bool},
    NULL,
    NULL,
    NULL,
    '1.0.0',
    'manual',
    NULL,
    NULL
)
""")

print(f"✅ Registro inserido com sucesso!")
print()
print(f"   Status: RUNNING")
print(f"   Ambiente: {env}")
print(f"   Run Date: {run_date}")
print(f"   Full Refresh: {full_refresh}")
print(f"   Start Time: {start_ts}")
print()
print("🔄 Este registro será atualizado no FINAL da execução")
print("🔄 Status final será: SUCCESS ou FAILED")
print()
print("="*60)
print()

## 🛠️ Bloco 9: Funções Utilitárias do Pipeline

### 📋 Descrição

Este bloco define **5 funções reutilizáveis** que serão usadas nos notebooks Bronze/Silver/Gold.

**Por que criar funções?**
* **Reutilização**: Código usado em múltiplos notebooks
* **Consistência**: Todos os notebooks seguem o mesmo padrão de log
* **Manutenção**: Alterar em 1 lugar propaga para todos

---

### 🛠️ Função 1: `log_api_request()`

**O que faz:** Registra chamadas HTTP na tabela `ops.ingestion_requests`.

**Uso típico:**
```python
response = requests.get(url)
log_api_request(
    endpoint=url,
    http_status=response.status_code,
    response_time_ms=response.elapsed.total_seconds() * 1000
)
```

---

### 🛠️ Função 2: `get_checkpoint()`

**O que faz:** Lê o último checkpoint da tabela `ops.checkpoints`.

**Uso típico:**
```python
ultimo_id = get_checkpoint(
    source_endpoint='deputados',
    checkpoint_type='max_id'
)
print(f"Processar a partir do ID: {ultimo_id}")
```

---

### 🛠️ Função 3: `update_checkpoint()`

**O que faz:** Atualiza checkpoint após ingestão bem-sucedida.

**Uso típico:**
```python
update_checkpoint(
    source_endpoint='deputados',
    checkpoint_type='max_id',
    checkpoint_value='12345'
)
```

---

### 🛠️ Função 4: `register_quality_check()`

**O que faz:** Registra resultado de regra QA em `log.quality_checks`.

**Uso típico:**
```python
register_quality_check(
    table_name='bronze.deputados_raw',
    rule_id='PK_NOT_NULL',
    rule_type='HARD',
    status='PASS'
)
```

---

### 🛠️ Função 5: `finalize_run()`

**O que faz:** Atualiza registro em `ops.ingestion_runs` com status final.

**Uso típico:**
```python
# No final do pipeline
finalize_run(
    status='SUCCESS',
    total_records=10000
)
```

In [0]:
# ============================================================
# FUNÇÃO 1: log_api_request
# ============================================================
# Registra chamadas HTTP na tabela ops.ingestion_requests.
# ============================================================

def log_api_request(endpoint, http_status, response_time_ms, 
                    request_params=None, error_message=None):
    """
    Registra uma chamada HTTP na tabela ops.ingestion_requests.
    
    Parâmetros:
        endpoint (str): URL do endpoint chamado
        http_status (int): Status HTTP da resposta (200, 404, 500, etc)
        response_time_ms (float): Tempo de resposta em milissegundos
        request_params (dict): Parâmetros da request (opcional)
        error_message (str): Mensagem de erro se houver (opcional)
    
    Retorna:
        str: request_id gerado
    """
    import uuid
    from datetime import datetime
    from pyspark.sql import Row
    
    # Gerar request_id único
    request_id = uuid.uuid4().hex
    
    # Criar DataFrame com o registro
    df_request = spark.createDataFrame([
        Row(
            request_id=request_id,
            load_id=load_id,
            endpoint=endpoint,
            http_method='GET',  # Assumindo GET por padrão
            http_status=http_status,
            response_time_ms=response_time_ms,
            request_ts=datetime.now(),
            request_params=str(request_params) if request_params else None,
            response_body=None,  # Não armazenar body por padrão (muito grande)
            error_message=error_message,
            retry_count=0
        )
    ])
    
    # Inserir na tabela
    tabela_requests = f"{catalog}.{schema_ops}.ingestion_requests"
    df_request.write.format("delta").mode("append").saveAsTable(tabela_requests)
    
    return request_id

print("✅ Função log_api_request() definida!")
print("   🎯 Uso: log_api_request(endpoint, http_status, response_time_ms)")
print()

In [0]:
# ============================================================
# FUNÇÃO 2: get_checkpoint
# ============================================================
# Lê o último checkpoint da tabela ops.checkpoints.
# ============================================================

def get_checkpoint(source_endpoint, checkpoint_type='max_id'):
    """
    Lê o último checkpoint para um endpoint específico.
    
    Parâmetros:
        source_endpoint (str): Nome do endpoint/entidade (ex: 'deputados')
        checkpoint_type (str): Tipo do checkpoint (ex: 'max_id', 'last_update_date')
    
    Retorna:
        str: Valor do checkpoint (None se não existir)
    """
    from pyspark.sql.functions import col
    
    tabela_checkpoints = f"{catalog}.{schema_ops}.checkpoints"
    
    # Tentar ler checkpoint
    try:
        df_checkpoint = spark.table(tabela_checkpoints) \
            .filter(
                (col("source_endpoint") == source_endpoint) & 
                (col("checkpoint_type") == checkpoint_type)
            ) \
            .orderBy(col("updated_ts").desc()) \
            .limit(1)
        
        # Se encontrou, retornar valor
        if df_checkpoint.count() > 0:
            checkpoint_value = df_checkpoint.first()["checkpoint_value"]
            print(f"📌 Checkpoint encontrado: {source_endpoint}.{checkpoint_type} = {checkpoint_value}")
            return checkpoint_value
        else:
            print(f"⚠️  Checkpoint não encontrado: {source_endpoint}.{checkpoint_type}")
            print(f"   🔄 Primeira execução ou full refresh")
            return None
            
    except Exception as e:
        print(f"⚠️  Erro ao ler checkpoint: {str(e)}")
        return None

print("✅ Função get_checkpoint() definida!")
print("   🎯 Uso: valor = get_checkpoint('deputados', 'max_id')")
print()

In [0]:
# ============================================================
# FUNÇÃO 3: update_checkpoint
# ============================================================
# Atualiza checkpoint após ingestão bem-sucedida.
# ============================================================

def update_checkpoint(source_endpoint, checkpoint_type, checkpoint_value):
    """
    Atualiza o checkpoint para um endpoint específico.
    
    Parâmetros:
        source_endpoint (str): Nome do endpoint/entidade
        checkpoint_type (str): Tipo do checkpoint
        checkpoint_value (str): Novo valor do checkpoint
    
    Retorna:
        bool: True se atualizado com sucesso
    """
    from datetime import datetime
    from pyspark.sql import Row
    from pyspark.sql.functions import col
    
    tabela_checkpoints = f"{catalog}.{schema_ops}.checkpoints"
    
    try:
        # Verificar se já existe checkpoint
        df_existe = spark.table(tabela_checkpoints) \
            .filter(
                (col("source_endpoint") == source_endpoint) & 
                (col("checkpoint_type") == checkpoint_type)
            )
        
        existe = df_existe.count() > 0
        
        if existe:
            # UPDATE: Usar MERGE para atualizar
            print(f"🔄 Atualizando checkpoint: {source_endpoint}.{checkpoint_type} = {checkpoint_value}")
            
            # Delta Lake MERGE
            from delta.tables import DeltaTable
            
            delta_table = DeltaTable.forName(spark, tabela_checkpoints)
            
            # Criar DataFrame com novo valor
            df_novo = spark.createDataFrame([
                Row(
                    source_endpoint=source_endpoint,
                    checkpoint_type=checkpoint_type,
                    checkpoint_value=str(checkpoint_value),
                    updated_ts=datetime.now(),
                    load_id=load_id
                )
            ])
            
            # MERGE
            delta_table.alias("target").merge(
                df_novo.alias("source"),
                "target.source_endpoint = source.source_endpoint AND target.checkpoint_type = source.checkpoint_type"
            ).whenMatchedUpdateAll().execute()
            
        else:
            # INSERT: Novo checkpoint
            print(f"➕ Criando novo checkpoint: {source_endpoint}.{checkpoint_type} = {checkpoint_value}")
            
            df_novo = spark.createDataFrame([
                Row(
                    source_endpoint=source_endpoint,
                    checkpoint_type=checkpoint_type,
                    checkpoint_value=str(checkpoint_value),
                    updated_ts=datetime.now(),
                    load_id=load_id
                )
            ])
            
            df_novo.write.format("delta").mode("append").saveAsTable(tabela_checkpoints)
        
        print(f"✅ Checkpoint atualizado com sucesso!")
        return True
        
    except Exception as e:
        print(f"❌ Erro ao atualizar checkpoint: {str(e)}")
        return False

print("✅ Função update_checkpoint() definida!")
print("   🎯 Uso: update_checkpoint('deputados', 'max_id', '12345')")
print()

In [0]:
# ============================================================
# FUNÇÃO 4: register_quality_check
# ============================================================
# Registra resultado de regra QA em log.quality_checks.
# ============================================================

def register_quality_check(table_name, rule_id, rule_type, status, 
                          observed_value=None, expected_value=None, 
                          rule_name=None, error_message=None):
    """
    Registra o resultado de uma checagem de qualidade.
    
    Parâmetros:
        table_name (str): Nome completo da tabela checada
        rule_id (str): ID da regra (ex: 'PK_NOT_NULL')
        rule_type (str): Tipo da regra ('HARD' ou 'SOFT')
        status (str): Status do check ('PASS' ou 'FAIL')
        observed_value: Valor observado (opcional)
        expected_value: Valor esperado (opcional)
        rule_name (str): Nome descritivo da regra (opcional)
        error_message (str): Mensagem de erro se falhou (opcional)
    
    Retorna:
        str: check_id gerado
    """
    import uuid
    from datetime import datetime
    from pyspark.sql import Row
    
    # Gerar check_id único
    check_id = uuid.uuid4().hex
    
    # Determinar layer a partir do table_name
    if '.bronze.' in table_name:
        layer = 'bronze'
    elif '.silver.' in table_name:
        layer = 'silver'
    elif '.gold.' in table_name:
        layer = 'gold'
    else:
        layer = 'unknown'
    
    # Criar DataFrame com o registro
    df_check = spark.createDataFrame([
        Row(
            check_id=check_id,
            load_id=load_id,
            layer=layer,
            table_name=table_name,
            entity_name=None,  # Pode ser extraído do table_name se necessário
            rule_id=rule_id,
            rule_name=rule_name,
            rule_type=rule_type,
            rule_category=None,
            status=status,
            observed_value=str(observed_value) if observed_value is not None else None,
            expected_value=str(expected_value) if expected_value is not None else None,
            deviation_value=None,
            execution_time_sec=None,
            error_message=error_message,
            check_ts=datetime.now()
        )
    ])
    
    # Inserir na tabela
    tabela_checks = f"{catalog}.{schema_log}.quality_checks"
    df_check.write.format("delta").mode("append").saveAsTable(tabela_checks)
    
    # Print feedback
    status_emoji = "✅" if status == 'PASS' else "❌"
    type_emoji = "🛑" if rule_type == 'HARD' else "⚠️"
    
    print(f"{status_emoji} {type_emoji} [{rule_type}] {rule_id}: {status}")
    if observed_value is not None:
        print(f"   Observado: {observed_value}")
    
    return check_id

print("✅ Função register_quality_check() definida!")
print("   🎯 Uso: register_quality_check(table, rule_id, 'HARD', 'PASS')")
print()

In [0]:
# ============================================================
# FUNÇÃO 5: finalize_run
# ============================================================
# Atualiza registro em ops.ingestion_runs com status final.
# ============================================================

def finalize_run(status, total_records=None, error_message=None, error_type=None):
    """
    Finaliza a execução atualizando o registro em ops.ingestion_runs.
    
    Parâmetros:
        status (str): Status final ('SUCCESS' ou 'FAILED')
        total_records (int): Total de registros processados (opcional)
        error_message (str): Mensagem de erro se falhou (opcional)
        error_type (str): Tipo do erro (opcional)
    
    Retorna:
        bool: True se atualizado com sucesso
    """
    from datetime import datetime
    from delta.tables import DeltaTable
    from pyspark.sql import Row
    from pyspark.sql.functions import col
    
    tabela_runs = f"{catalog}.{schema_ops}.ingestion_runs"
    
    try:
        # Calcular tempo de execução
        end_ts = datetime.now()
        execution_time_sec = (end_ts - start_ts).total_seconds()
        
        print("="*60)
        print("🎯 FINALIZANDO EXECUÇÃO")
        print("="*60)
        print()
        print(f"🆔 Load ID: {load_id}")
        print(f"⏰ Duração: {execution_time_sec:.2f} segundos")
        print(f"📊 Status: {status}")
        
        if total_records:
            print(f"📄 Registros: {total_records:,}")
        
        if error_message:
            print(f"❌ Erro: {error_message}")
        
        print()
        
        # Delta Lake MERGE para atualizar
        delta_table = DeltaTable.forName(spark, tabela_runs)
        
        # Criar DataFrame com valores atualizados
        df_update = spark.createDataFrame([
            Row(
                load_id=load_id,
                end_ts=end_ts,
                execution_time_sec=execution_time_sec,
                status=status,
                total_records_ingested=total_records,
                error_message=error_message,
                error_type=error_type
            )
        ])
        
        # MERGE
        delta_table.alias("target").merge(
            df_update.alias("source"),
            "target.load_id = source.load_id"
        ).whenMatchedUpdate(
            set={
                "end_ts": "source.end_ts",
                "execution_time_sec": "source.execution_time_sec",
                "status": "source.status",
                "total_records_ingested": "source.total_records_ingested",
                "error_message": "source.error_message",
                "error_type": "source.error_type"
            }
        ).execute()
        
        print("✅ Registro atualizado em ops.ingestion_runs")
        print()
        print("="*60)
        print()
        
        return True
        
    except Exception as e:
        print(f"❌ Erro ao finalizar execução: {str(e)}")
        return False

print("✅ Função finalize_run() definida!")
print("   🎯 Uso: finalize_run('SUCCESS', total_records=10000)")
print()
print("🎉 BLOCO 9 COMPLETO: Funções utilitárias criadas!")
print()

## ✅ Bloco 10: Validação da Infraestrutura

### 📋 Descrição

Este bloco **valida** que toda a infraestrutura foi criada corretamente.

**O que faz?**
* Lista todas as tabelas em cada schema
* Conta quantidade de tabelas por schema
* Exibe resumo da inicialização
* Confirma que tudo está pronto para uso

**Por que é importante?**
* Confirma que TODAS as 15 tabelas foram criadas
* Identifica falhas na criação de alguma tabela
* Fornece resumo visual da arquitetura

**O que esperar:**
```
✅ bronze: 0 tabelas (serão criadas nos notebooks Bronze)
✅ silver: 0 tabelas (serão criadas nos notebooks Silver)
✅ gold: 0 tabelas (serão criadas nos notebooks Gold)
✅ ops: 4 tabelas (ingestion_runs, ingestion_requests, checkpoints, dead_letter)
✅ log: 4 tabelas (bronze_ingest_runs, silver_transform_runs, gold_mart_runs, quality_checks)
✅ qa: 3 tabelas (quality_results, quality_metrics, rejects)
```

---

### 👀 Próximos Passos

Após executar este notebook com sucesso:

1. **Notebooks Bronze**: Criar notebooks de ingestão por entidade
   * `01_bronze_deputados.py`
   * `01_bronze_frentes.py`
   * `01_bronze_eventos.py`
   * `01_bronze_votacoes.py`
   * `01_bronze_ceap.py`
   * `01_bronze_cpis.py`
   * `01_bronze_proposicoes.py`

2. **Notebooks Silver**: Criar notebooks de transformação

3. **Notebooks Gold**: Criar marts analíticos

4. **Orquestração**: Configurar Databricks Jobs

In [0]:
# ============================================================
# VALIDAÇÃO FINAL DA INFRAESTRUTURA
# ============================================================
# Lista todas as tabelas criadas em cada schema.
# Exibe resumo final da inicialização.
# ============================================================

from pyspark.sql.functions import count, col

print("="*60)
print("✅ VALIDANDO INFRAESTRUTURA DO PIPELINE")
print("="*60)
print()

# Lista de schemas para validar
schemas = {
    'bronze': schema_bronze,
    'silver': schema_silver,
    'gold': schema_gold,
    'ops': schema_ops,
    'log': schema_log,
    'qa': schema_qa
}

total_tabelas = 0

# Iterar por cada schema
for nome_schema, schema_var in schemas.items():
    print(f"📁 Schema: {catalog}.{schema_var}")
    print("-" * 60)
    
    # Listar tabelas do schema
    df_tables = spark.sql(f"SHOW TABLES IN {catalog}.{schema_var}")
    
    # Contar tabelas
    qtd_tabelas = df_tables.count()
    total_tabelas += qtd_tabelas
    
    if qtd_tabelas > 0:
        print(f"   ✅ {qtd_tabelas} tabela(s) encontrada(s):")
        print()
        
        # Exibir cada tabela
        tabelas = df_tables.collect()
        for row in tabelas:
            table_name = row['tableName']
            print(f"      • {table_name}")
        print()
    else:
        print(f"   ⚠️  0 tabelas (serão criadas posteriormente)")
        print()

print("="*60)
print("📊 RESUMO DA INICIALIZAÇÃO")
print("="*60)
print()
print(f"✅ Catálogo: {catalog}")
print(f"✅ Schemas: 6 (bronze, silver, gold, ops, log, qa)")
print(f"✅ Tabelas de controle: {total_tabelas}")
print(f"✅ Load ID gerado: {load_id}")
print(f"✅ Execução registrada em: ops.ingestion_runs")
print()
print("🔧 Estrutura esperada:")
print("   • ops: 4 tabelas (ingestion_runs, ingestion_requests, checkpoints, dead_letter)")
print("   • log: 4 tabelas (bronze_ingest_runs, silver_transform_runs, gold_mart_runs, quality_checks)")
print("   • qa: 3 tabelas (quality_results, quality_metrics, rejects)")
print("   • bronze/silver/gold: Tabelas criadas nos notebooks específicos")
print()
print("🎯 Próximos passos:")
print("   1. Criar notebooks Bronze para cada entidade")
print("   2. Criar notebooks Silver para transformações")
print("   3. Criar notebooks Gold para marts analíticos")
print("   4. Configurar Databricks Jobs para orquestração")
print()
print("="*60)
print("🎉 INICIALIZAÇÃO CONCLUÍDA COM SUCESSO!")
print("="*60)
print()
print(f"🚀 Pipeline Fast Track pronto para uso!")
print(f"📌 Ambiente: {env}")
print(f"📅 Run Date: {run_date}")
print()